In [1]:
# Install libraries 
 
import subprocess, sys
pkgs = ["librosa", "soundfile", "tqdm", "pandas", "numpy",
        "scikit-learn", "matplotlib", "seaborn",
        "tensorflow", "opencv-python", "onnxruntime"]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("All packages installed")

All packages installed


In [2]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.export(format="onnx", opset=17)

WARNING Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.pt... [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4192)
Ultralytics 8.4.72  Python-3.9.25 torch-2.8.0+cpu CPU (AMD Ryzen 7 5800U with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from 'yolo11n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.4 MB)

ONNX: starting export with onnx 1.19.1 opset 17...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  4.5s, saved as 'yolo11n.onnx' (10.2 MB)

Export complete (5.2s)
Results saved to C:\Users\Acer Swift X\Downloads\iot project\ML-IOT-Project-2-Smart-Security-System\yolo11n.onnx
Predict:         yolo predict task=detect model=yolo11n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo11n.onnx imgsz=640 data=/usr/src/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'yolo11n.onnx'

In [3]:
# Dataset Preparation and Combination 

import os, random
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

BASE             = r"C:\Users\Acer Swift X\Downloads\iot project"
ESC50_AUDIO_DIR  = rf"{BASE}\audio"
ESC50_META_CSV   = rf"{BASE}\esc50.csv"
US8K_AUDIO_DIR   = rf"{BASE}\audio_urban"
US8K_META_CSV    = rf"{BASE}\UrbanSound8K.csv"
SCREAMING_DIR    = rf"{BASE}\Screaming"
OUTPUT_DIR       = rf"{BASE}\combined_dataset"
OUTPUT_AUDIO_DIR = os.path.join(OUTPUT_DIR, "audio")
OUTPUT_META_CSV  = os.path.join(OUTPUT_DIR, "metadata.csv")

SAMPLE_RATE   = 22050
CLIP_DURATION = 4.0
RANDOM_SEED   = 42


MAX_NORMAL_PER_DATASET = 400

ESC50_NORMAL_CATS = [
    "footsteps", "door_wood_knock", "vacuum_cleaner",
    "dog", "clock_tick", "laughing", "washing_machine", "keyboard_typing",
]
US8K_NORMAL_CLASSES = {
    2: "children_playing",
    3: "dog_bark",
    9: "street_music",
    5: "engine_idling",
    4: "drilling",
    0: "air_conditioner",
}

MAX_SCREAMING = 800

LABEL_MAP = {
    "glass_breaking": 0,
    "clock_alarm"   : 1,
    "gun_shot"      : 2,
    "siren"         : 3,
    "screaming"     : 4,
    "normal"        : 5,
    "silence"       : 6,
}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_AUDIO_DIR, exist_ok=True)
records = []

def load_and_normalise(path, sr=SAMPLE_RATE, duration=CLIP_DURATION):
    y, _ = librosa.load(path, sr=sr, mono=True)
    target_len = int(sr * duration)
    if len(y) > target_len:
        start = random.randint(0, len(y) - target_len)
        y = y[start:start + target_len]
    else:
        y = np.pad(y, (0, target_len - len(y)))
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    return y

def save_clip(y, dest_path, sr=SAMPLE_RATE):
    sf.write(dest_path, y, sr)

def add_record(filename, label, source):
    records.append({"filename": filename, "label": label,
                    "label_id": LABEL_MAP[label], "source": source,
                    "duration": CLIP_DURATION})


# ESC-50
print("\nProcessing ESC-50")

ESC50_DANGER_CATS = {
    "glass_breaking": "glass_breaking",
    "clock_alarm"   : "clock_alarm",
}

# Spread the MAX_NORMAL_PER_DATASET quota evenly across normal categories
quota_per_cat_esc = MAX_NORMAL_PER_DATASET // len(ESC50_NORMAL_CATS)
normal_cat_counts_esc = {c: 0 for c in ESC50_NORMAL_CATS}

esc_meta = pd.read_csv(ESC50_META_CSV)
normal_count_esc = 0

for _, row in tqdm(esc_meta.iterrows(), total=len(esc_meta), desc="ESC-50"):
    cat = row["category"]

    if cat in ESC50_DANGER_CATS:
        label = ESC50_DANGER_CATS[cat]
    elif cat in ESC50_NORMAL_CATS:
        if normal_cat_counts_esc[cat] >= quota_per_cat_esc:
            continue
        label = "normal"
        normal_cat_counts_esc[cat] += 1
        normal_count_esc += 1
    else:
        continue  

    src = os.path.join(ESC50_AUDIO_DIR, row["filename"])
    if not os.path.exists(src):
        print(f"  [WARN] Missing: {src}"); continue

    y = load_and_normalise(src)
    fname = f"esc50_{label}_{cat}_{row['filename']}"
    save_clip(y, os.path.join(OUTPUT_AUDIO_DIR, fname))
    add_record(fname, label, "ESC-50")

print(f"Records: {len(records)} (normal: {normal_count_esc})")
print(f"\nNormal breakdown: {normal_cat_counts_esc}")


# UrbanSound8K:
print("\nProcessing UrbanSound8K")

US8K_DANGER_MAP = {6: "gun_shot", 8: "siren"}

quota_per_cat_us8k = MAX_NORMAL_PER_DATASET // len(US8K_NORMAL_CLASSES)
normal_cat_counts_us8k = {name: 0 for name in US8K_NORMAL_CLASSES.values()}

us8k_meta = pd.read_csv(US8K_META_CSV)
normal_count_us8k = 0

for _, row in tqdm(us8k_meta.iterrows(), total=len(us8k_meta), desc="US8K"):
    cls = row["classID"]

    if cls in US8K_DANGER_MAP:
        label = US8K_DANGER_MAP[cls]
    elif cls in US8K_NORMAL_CLASSES:
        cat_name = US8K_NORMAL_CLASSES[cls]
        if normal_cat_counts_us8k[cat_name] >= quota_per_cat_us8k:
            continue
        label = "normal"
        normal_cat_counts_us8k[cat_name] += 1
        normal_count_us8k += 1
    else:
        continue  

    src = os.path.join(US8K_AUDIO_DIR, f"fold{row['fold']}", row["slice_file_name"])
    if not os.path.exists(src):
        print(f"  [WARN] Missing: {src}"); continue

    y = load_and_normalise(src)
    fname = f"us8k_{label}_{cat_name if label=='normal' else label}_fold{row['fold']}_{row['slice_file_name']}"
    save_clip(y, os.path.join(OUTPUT_AUDIO_DIR, fname))
    add_record(fname, label, "UrbanSound8K")

print(f"Records: {len(records)} (normal: {normal_count_us8k})")
print(f"\nNormal breakdown: {normal_cat_counts_us8k}")


# Screaming dataset 
print("\nProcessing Screaming dataset")
scream_files = list(Path(SCREAMING_DIR).rglob("*.wav")) + \
               list(Path(SCREAMING_DIR).rglob("*.WAV"))

# Shuffle and cap to MAX_SCREAMING
random.shuffle(scream_files)
scream_files = scream_files[:MAX_SCREAMING]

for i, p in enumerate(tqdm(scream_files, desc="Screaming")):
    y = load_and_normalise(str(p))
    fname = f"scream_{i:04d}_{p.name}"
    save_clip(y, os.path.join(OUTPUT_AUDIO_DIR, fname))
    add_record(fname, "screaming", "HumanScreamingDS")
print(f"Records: {len(records)}  (screaming used: {len(scream_files)})")


# Silence (synthetic) 
print("\nGenerating silence clips")
target_len = int(SAMPLE_RATE * CLIP_DURATION)
for i in tqdm(range(200), desc="Silence"):
    y = np.random.normal(0, random.uniform(1e-5, 5e-4), target_len).astype(np.float32)
    fname = f"silence_{i:04d}.wav"
    save_clip(y, os.path.join(OUTPUT_AUDIO_DIR, fname))
    add_record(fname, "silence", "synthetic")
print(f"Records: {len(records)}")


# Save metadata 
df = pd.DataFrame(records)
df.to_csv(OUTPUT_META_CSV, index=False)
print("\n=== Class Distribution ===")
print(df["label"].value_counts().to_string())
print(f"\nTotal clips : {len(df)}")



Processing ESC-50


ESC-50:   0%|          | 0/2000 [00:00<?, ?it/s]c:\Users\Acer Swift X\miniconda3\envs\tfgpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ESC-50: 100%|██████████| 2000/2000 [00:16<00:00, 122.90it/s]


Records: 400 (normal: 320)

Normal breakdown: {'footsteps': 40, 'door_wood_knock': 40, 'vacuum_cleaner': 40, 'dog': 40, 'clock_tick': 40, 'laughing': 40, 'washing_machine': 40, 'keyboard_typing': 40}

Processing UrbanSound8K


US8K: 100%|██████████| 8732/8732 [00:53<00:00, 162.08it/s] 


Records: 2099 (normal: 396)

Normal breakdown: {'children_playing': 66, 'dog_bark': 66, 'street_music': 66, 'engine_idling': 66, 'drilling': 66, 'air_conditioner': 66}

Processing Screaming dataset


Screaming: 100%|██████████| 800/800 [00:29<00:00, 27.17it/s]


Records: 2899  (screaming used: 800)

Generating silence clips


Silence: 100%|██████████| 200/200 [00:02<00:00, 83.52it/s]

Records: 3099

=== Class Distribution ===
label
siren             929
screaming         800
normal            716
gun_shot          374
silence           200
clock_alarm        40
glass_breaking     40

Total clips : 3099


In [5]:
# Feature Extraction

import cv2

METADATA_CSV    = r"C:\Users\Acer Swift X\Downloads\iot project\combined_dataset\metadata.csv"
AUDIO_DIR       = r"C:\Users\Acer Swift X\Downloads\iot project\combined_dataset\audio"
FEATURES_NPZ    = r"C:\Users\Acer Swift X\Downloads\iot project\combined_dataset\features.npz"

SAMPLE_RATE     = 22050
DURATION        = 4.0        
N_MFCC          = 40           # also extracted for lightweight models
N_MELS          = 128
HOP_LENGTH      = 512
N_FFT           = 2048
IMG_SIZE        = (224, 224)   # MobileNetV2 input


def extract_melspec(y: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalise to 0–1
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    mel_norm = mel_norm.astype(np.float32)

    # Resize to IMG_SIZE
    img = cv2.resize(mel_norm, IMG_SIZE, interpolation=cv2.INTER_LINEAR)

    # Stack to 3 channels (224, 224, 3)
    img_3ch = np.stack([img, img, img], axis=-1)  
    return img_3ch


def extract_mfcc_vector(y: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    feat = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        mfcc.max(axis=1),
    ])
    return feat.astype(np.float32)


def main():
    df = pd.read_csv(METADATA_CSV)
    n  = len(df)

    X_img   = np.zeros((n, 224, 224, 3), dtype=np.float32)
    X_mfcc  = np.zeros((n, N_MFCC * 3),  dtype=np.float32)
    y_labels = np.zeros(n, dtype=np.int32)

    print(f"Extracting features for {n} clips")
    failed = 0

    for i, row in tqdm(df.iterrows(), total=n):
        path = os.path.join(AUDIO_DIR, row["filename"])
        try:
            y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
            target_len = int(SAMPLE_RATE * DURATION)
            if len(y) > target_len:
                y = y[:target_len]
            else:
                y = np.pad(y, (0, target_len - len(y)))

            X_img[i]  = extract_melspec(y)
            X_mfcc[i] = extract_mfcc_vector(y)
            y_labels[i] = int(row["label_id"])

        except Exception as e:
            print(f"  [WARN] {row['filename']}: {e}")
            failed += 1

    print(f"\nFeature extraction complete.  Failed: {failed}/{n}")
    print(f"X_img  shape : {X_img.shape}")   
    print(f"X_mfcc shape : {X_mfcc.shape}")  
    print(f"Labels shape : {y_labels.shape}")

    np.savez_compressed(
        FEATURES_NPZ,
        X_img=X_img,
        X_mfcc=X_mfcc,
        y=y_labels,
    )
    print(f"Saved to {FEATURES_NPZ} ")


if __name__ == "__main__":
    main()

Extracting features for 3099 clips


100%|██████████| 3099/3099 [01:25<00:00, 36.43it/s]



Feature extraction complete.  Failed: 0/3099
X_img  shape : (3099, 224, 224, 3)
X_mfcc shape : (3099, 120)
Labels shape : (3099,)
Saved to C:\Users\Acer Swift X\Downloads\iot project\combined_dataset\features.npz 


In [ ]:
# MobileNetV2 Audio Classifier — Training

# Suppress TensorFlow Warnings
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)


import matplotlib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import tensorflow as tf

# Hide TensorFlow logger messages
tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)

# Enable GPU memory growth (prevents TensorFlow from allocating all GPU memory)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Using GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("Running on CPU")

from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical

FEATURES_NPZ   = r"C:\Users\Acer Swift X\Downloads\iot project\combined_dataset\features.npz"
MODEL_DIR      = "models"
MODEL_PATH     = os.path.join(MODEL_DIR, "mobilenetv2_audio.h5")
TFLITE_PATH    = os.path.join(MODEL_DIR, "mobilenetv2_audio.tflite")
PLOT_DIR       = "plots"

NUM_CLASSES    = 7
IMG_SIZE       = (224, 224, 3)
BATCH_SIZE     = 32
EPOCHS         = 20

RANDOM_SEED    = 42
TEST_SPLIT     = 0.15
VAL_SPLIT      = 0.15

CLASS_NAMES = [
    "glass_breaking", "clock_alarm", "gun_shot",
    "siren", "screaming", "normal","silence"
]


os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR,  exist_ok=True)
tf.random.set_seed(RANDOM_SEED)


# Load features 
print("Loading features: ")
data = np.load(FEATURES_NPZ)
X    = data["X_img"].astype(np.float32)    # (N, 224, 224, 3)  already 0-1
y    = data["y"].astype(np.int32)          # (N,)

print(f"X shape : {X.shape}")
print(f"\nClasses : {np.bincount(y)}")

y_cat = to_categorical(y, NUM_CLASSES)

# Stratified split: 70% train / 15% val / 15% test
X_tmp,  X_test,  y_tmp,  y_test  = train_test_split(
    X, y_cat, test_size=TEST_SPLIT, stratify=y, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val  = train_test_split(
    X_tmp, y_tmp, test_size=VAL_SPLIT/(1-TEST_SPLIT),
    stratify=y_tmp.argmax(1), random_state=RANDOM_SEED)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


# Data augmentation 
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),          # mirrors spectrogram in time
    layers.RandomTranslation(0.05, 0.05),     # small time/freq shift
    layers.GaussianNoise(0.01),               # adds light noise
], name="augmentation")


# Build model 
print("\nBuilding MobileNetV2 model ")

base_model = MobileNetV2(
    input_shape=IMG_SIZE,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False   

inputs  = tf.keras.Input(shape=IMG_SIZE)
x       = augment(inputs, training=True)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation="relu")(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(128, activation="relu")(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs, name="MobileNetV2_AudioClassifier")
model.summary()


# Train head 
print("\nTraining model")

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cb_list = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True,
                            monitor="val_accuracy"),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3,
                                monitor="val_loss", verbose=1),
    callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True,
                              monitor="val_accuracy", verbose=1),
]

hist1 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb_list,
    verbose=1,
)


Running on CPU
Loading features: 
X shape : (3099, 224, 224, 3)

Classes : [ 40  40 374 929 800 716 200]

Train: 2169 | Val: 465 | Test: 465

Building MobileNetV2 model 


Model: "MobileNetV2_AudioClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,624,839 (10.01 MB)

 Trainable params: 364,295 (1.39 MB)

...

68/68 ━━━━━━━━━━━━━━━━━━━━ 32s 403ms/step - accuracy: 0.6522 - loss: 1.0674 - val_accuracy: 0.8323 - val_loss: 0.5336 - learning_rate: 0.0010
Epoch 2/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step - accuracy: 0.8465 - loss: 0.4317
Epoch 2: val_accuracy improved from 0.83226 to 0.88817, saving model to models\mobilenetv2_audio.h5


68/68 ━━━━━━━━━━━━━━━━━━━━ 28s 413ms/step - accuracy: 0.8465 - loss: 0.4319 - val_accuracy: 0.8882 - val_loss: 0.3840 - learning_rate: 0.0010
Epoch 3/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 348ms/step - accuracy: 0.8767 - loss: 0.3391
Epoch 3: val_accuracy improved from 0.88817 to 0.90753, saving model to models\mobilenetv2_audio.h5


68/68 ━━━━━━━━━━━━━━━━━━━━ 28s 416ms/step - accuracy: 0.8766 - loss: 0.3396 - val_accuracy: 0.9075 - val_loss: 0.3173 - learning_rate: 0.0010
Epoch 4/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 345ms/step - accuracy: 0.8890 - loss: 0.2949
Epoch 4: val_accuracy improved from 0.90753 to 0.90968, saving model to models\mobilenetv2_audio.h5


68/68 ━━━━━━━━━━━━━━━━━━━━ 28s 413ms/step - accuracy: 0.8890 - loss: 0.2954 - val_accuracy: 0.9097 - val_loss: 0.2870 - learning_rate: 0.0010
Epoch 5/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step - accuracy: 0.9182 - loss: 0.2462
Epoch 5: val_accuracy improved from 0.90968 to 0.91183, saving model to models\mobilenetv2_audio.h5


68/68 ━━━━━━━━━━━━━━━━━━━━ 29s 420ms/step - accuracy: 0.9181 - loss: 0.2464 - val_accuracy: 0.9118 - val_loss: 0.2953 - learning_rate: 0.0010
Epoch 6/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 385ms/step - accuracy: 0.9057 - loss: 0.2327
Epoch 6: val_accuracy did not improve from 0.91183
68/68 ━━━━━━━━━━━━━━━━━━━━ 30s 448ms/step - accuracy: 0.9057 - loss: 0.2327 - val_accuracy: 0.9097 - val_loss: 0.2870 - learning_rate: 0.0010
Epoch 7/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step - accuracy: 0.9062 - loss: 0.2488
Epoch 7: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 7: val_accuracy did not improve from 0.91183
68/68 ━━━━━━━━━━━━━━━━━━━━ 30s 444ms/step - accuracy: 0.9063 - loss: 0.2486 - val_accuracy: 0.9075 - val_loss: 0.3101 - learning_rate: 0.0010
Epoch 8/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 375ms/step - accuracy: 0.9254 - loss: 0.1902
Epoch 8: val_accuracy did not improve from 0.91183
68/68 ━━━━━━━━━━━━━━━━━━━━ 31s 456ms/step - accuracy: 0.9254 - loss: 0.1902 - val_accu

In [12]:
# Evaluate on test set

print("Evaluating on test set")
model.load_weights(MODEL_PATH)
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"    Test accuracy : {acc*100:.2f}%")
print(f"    Test loss     : {loss:.4f}")

y_pred  = model.predict(X_test).argmax(axis=1)
y_true  = y_test.argmax(axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix plot
cm  = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix – MobileNetV2 Audio Classifier")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "confusion_matrix.png"), dpi=150)
plt.close()
print(f"Confusion matrix saved to {PLOT_DIR}/confusion_matrix.png")

# Training curves plot
all_acc     = hist1.history["accuracy"]
all_val_acc = hist1.history["val_accuracy"]
all_loss    = hist1.history["loss"]
all_val_loss= hist1.history["val_loss"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(all_acc, label="Train"); ax1.plot(all_val_acc, label="Val")
ax1.set_title("Accuracy"); ax1.legend(); ax1.set_xlabel("Epoch"); ax1.set_ylim(0, 1)

ax2.plot(all_loss, label="Train"); ax2.plot(all_val_loss, label="Val")
ax2.set_title("Loss"); ax2.legend(); ax2.set_xlabel("Epoch");ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "training_curves.png"), dpi=150)
plt.close()
print(f"Training curves saved to {PLOT_DIR}/training_curves.png")

Evaluating on test set
    Test accuracy : 88.82%
    Test loss     : 0.3343
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 268ms/step

Classification Report:
                precision    recall  f1-score   support

glass_breaking       0.29      0.33      0.31         6
   clock_alarm       1.00      0.83      0.91         6
      gun_shot       0.90      0.93      0.91        56
         siren       0.98      0.88      0.93       139
     screaming       0.87      0.91      0.89       120
        normal       0.81      0.86      0.83       108
       silence       1.00      1.00      1.00        30

      accuracy                           0.89       465
     macro avg       0.83      0.82      0.83       465
  weighted avg       0.89      0.89      0.89       465

Confusion matrix saved to plots/confusion_matrix.png
Training curves saved to plots/training_curves.png


In [ ]:
# Export TFLite (for ESP32 / edge)

print("Exporting TFLite model ...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # INT8 quantisation
tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f"TFLite model saved to {TFLITE_PATH}  ({size_kb:.1f} KB)")
print("\nTraining complete")

Exporting TFLite model ...


INFO:tensorflow:Assets written to: C:\Users\ACERSW~1\AppData\Local\Temp\tmpqxsagord\assets


INFO:tensorflow:Assets written to: C:\Users\ACERSW~1\AppData\Local\Temp\tmpqxsagord\assets


Saved artifact at 'C:\Users\ACERSW~1\AppData\Local\Temp\tmpqxsagord'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_28')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  1921028648800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921028579568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921028579744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921028698528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921028701344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921028582032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921009483840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921009484720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921028582208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1921028583088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  

In [10]:
# Load ONNX + run detection 

import onnxruntime as ort
import numpy as np

BASE      = r"C:\Users\Acer Swift X\Downloads\iot project"
YOLO_ONNX = r"C:\Users\Acer Swift X\Downloads\iot project\ML-IOT-Project-2-Smart-Security-System\models\yolo11n.onnx"   # ultralytics saves it in the current folder

# CONFIG 
CONFIDENCE_THRESH = 0.45
NMS_IOU_THRESH    = 0.45
PERSON_CLASS_ID   = 0
IMG_SIZE          = 640
PRESENCE_MIN_AREA = 0.02

# Load ONNX session 
session    = ort.InferenceSession(YOLO_ONNX, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
print(f"Model loaded, input: {input_name}")
print(f"Output names: {[o.name for o in session.get_outputs()]}")


# Preprocessing
def preprocess(frame_bgr, size=IMG_SIZE):
    h, w   = frame_bgr.shape[:2]
    scale  = min(size/h, size/w)
    nh, nw = int(h*scale), int(w*scale)
    resized = cv2.resize(frame_bgr, (nw, nh))
    canvas  = np.full((size, size, 3), 114, dtype=np.uint8)
    ph, pw  = (size-nh)//2, (size-nw)//2
    canvas[ph:ph+nh, pw:pw+nw] = resized
    blob = canvas[:,:,::-1].astype(np.float32) / 255.0
    blob = np.expand_dims(blob.transpose(2,0,1), axis=0)  # (1,3,640,640)
    return blob, scale, (pw, ph)


# Postprocessing (YOLOv11 ONNX output: (1, 84, 8400)) 
def postprocess(output, frame_h, frame_w, scale, padding):
    # output shape: (1, 84, 8400), transpose to (8400, 84)
    preds = output[0].squeeze(0).T   
    pw, ph = padding

    # columns
    box_coords  = preds[:, :4]
    class_confs = preds[:, 4:]
    person_conf = class_confs[:, PERSON_CLASS_ID]

    mask = person_conf >= CONFIDENCE_THRESH
    if not mask.any():
        return [], []

    boxes   = box_coords[mask]
    scores  = person_conf[mask]

    # cx,cy,w,h → x1,y1,x2,y2 (letterbox coords)
    x1 = (boxes[:,0] - boxes[:,2]/2 - pw) / scale
    y1 = (boxes[:,1] - boxes[:,3]/2 - ph) / scale
    x2 = (boxes[:,0] + boxes[:,2]/2 - pw) / scale
    y2 = (boxes[:,1] + boxes[:,3]/2 - ph) / scale

    x1 = np.clip(x1, 0, frame_w); x2 = np.clip(x2, 0, frame_w)
    y1 = np.clip(y1, 0, frame_h); y2 = np.clip(y2, 0, frame_h)

    bboxes_cv = np.stack([x1, y1, x2-x1, y2-y1], axis=1).tolist()
    indices   = cv2.dnn.NMSBoxes(bboxes_cv, scores.tolist(),
                                  CONFIDENCE_THRESH, NMS_IOU_THRESH)
    if len(indices) == 0:
        return [], []

    final_boxes  = [(int(x1[i]),int(y1[i]),int(x2[i]),int(y2[i])) for i in indices.flatten()]
    final_scores = [float(scores[i]) for i in indices.flatten()]
    return final_boxes, final_scores


# Presence score 
def compute_presence_score(boxes, scores, frame_h, frame_w):
    if not boxes:
        return 0.0
    frame_area = frame_h * frame_w
    areas = [((x2-x1)*(y2-y1))/(frame_area+1e-6) for x1,y1,x2,y2 in boxes]
    # filter tiny detections
    valid = [(s, a) for s, a in zip(scores, areas) if a >= PRESENCE_MIN_AREA]
    if not valid:
        return 0.0
    mean_conf = np.mean([v[0] for v in valid])
    mean_area = np.mean([v[1] for v in valid])
    n = len(valid)
    return float(np.clip(mean_conf * (n**0.5) * (1 + mean_area), 0, 1))


# Draw boxes 
def draw_detections(frame, boxes, scores):
    out = frame.copy()
    for (x1,y1,x2,y2), score in zip(boxes, scores):
        cv2.rectangle(out, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(out, f"person {score:.2f}", (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 2)
    return out


# Quick test on a static image 
def run_on_image(img_path):
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"Cannot read: {img_path}"); return
    h, w = frame.shape[:2]
    blob, scale, padding = preprocess(frame)
    raw    = session.run(None, {input_name: blob})
    boxes, scores = postprocess(raw, h, w, scale, padding)
    presence = compute_presence_score(boxes, scores, h, w)
    print(f"Persons detected : {len(boxes)}")
    print(f"Presence score   : {presence:.3f}")
    result = draw_detections(frame, boxes, scores)
    cv2.imwrite("detection_result.jpg", result)
    print("Saved to detection_result.jpg")

run_on_image(r"C:\Users\Acer Swift X\Downloads\iot project\Shot 13 - Gemini_Generated_Image_tbxzl5tbxzl5tbxz.png")

Model loaded, input: images
Output names: ['output0']
Persons detected : 2
Presence score   : 1.000
Saved to detection_result.jpg


In [11]:
# Real-time camera detection 

from collections import deque

def run_human_detection(camera_source=0):

    cap = cv2.VideoCapture(camera_source)

    if not cap.isOpened():
        print("Cannot open camera — check CAMERA_SOURCE")
        return

    print("Camera opened. Press Q to quit")

    score_history = deque(maxlen=5)

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Failed to grab frame")
            break

        h, w = frame.shape[:2]

        blob, scale, padding = preprocess(frame)
        raw = session.run(None, {input_name: blob})

        boxes, scores = postprocess(raw, h, w, scale, padding)

        presence = compute_presence_score(
            boxes, scores, h, w
        )

        status = "Human Detected" if presence > 0.5 else "No Human"

        result = draw_detections(frame, boxes, scores)

        cv2.putText(result, f"Persons: {len(boxes)}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

        cv2.putText(result, f"Presence: {presence:.2f}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

        cv2.putText(result, f"Result: {status}", (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

        cv2.imshow("YOLO11 Human Detection", result)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Camera closed")

run_human_detection() # add ESP32-CAM

Camera opened. Press Q to quit
Camera closed
